In [ ]:
%matplotlib inline

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import vix_tcn_revin_xai_plus as vtrx

plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["axes.grid"] = True

CSV_PATH = ROOT / "data" / "raw" / "timeseries_data.csv"
INDEX_COL = "날짜"
TARGET_COL = "VIX"
DROP_COLS = ["Silver", "Copper", "USD/GBP", "USD/CNY", "USD/JPY", "USD/EUR", "USD/CAD"]

In [ ]:
df_raw = vtrx.load_frame(CSV_PATH, INDEX_COL, DROP_COLS)

print(f"shape      : {df_raw.shape}")
print(f"date range : {df_raw.index.min()} ~ {df_raw.index.max()}")
print(f"columns    : {list(df_raw.columns)}")
df_raw.head()

In [ ]:
summary_df = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "n_missing": df_raw.isna().sum(),
    "n_unique": df_raw.nunique(),
})
summary_df

In [ ]:
missing_df = df_raw.isna().sum().sort_values(ascending=False).to_frame("missing_count")
missing_df[missing_df["missing_count"] > 0].head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_raw.index, df_raw[TARGET_COL], lw=1.2)
ax.set_title(f"{TARGET_COL} level over time")
ax.set_xlabel("Date")
ax.set_ylabel(TARGET_COL)
plt.tight_layout()
plt.show()

In [ ]:
feature_candidates = ["VIX", "SPX", "Gold", "WTI", "DXY", "KOSPI"]
features_to_plot = [c for c in feature_candidates if c in df_raw.columns][:5]

fig, axes = plt.subplots(len(features_to_plot), 1, figsize=(14, 2.5 * len(features_to_plot)), sharex=True)
if len(features_to_plot) == 1:
    axes = [axes]

for ax, col in zip(axes, features_to_plot):
    ax.plot(df_raw.index, df_raw[col], lw=1)
    ax.set_title(col)
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
corr = df_raw[numeric_cols].corr()

top_corr = corr[TARGET_COL].sort_values(ascending=False).to_frame("corr_with_target")
top_corr.head(15)

In [ ]:
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
corr = df_raw[numeric_cols].corr()

top_corr = corr[TARGET_COL].sort_values(ascending=False).to_frame("corr_with_target")
top_corr.head(15)

In [ ]:
rolling_window = 20
eda_df = df_raw[[TARGET_COL]].copy()
eda_df[f"{TARGET_COL}_rolling_mean"] = eda_df[TARGET_COL].rolling(rolling_window).mean()
eda_df[f"{TARGET_COL}_rolling_std"] = eda_df[TARGET_COL].rolling(rolling_window).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(eda_df.index, eda_df[TARGET_COL], label=TARGET_COL, lw=1)
axes[0].plot(eda_df.index, eda_df[f"{TARGET_COL}_rolling_mean"], label="rolling mean", lw=1)
axes[0].legend()
axes[0].set_title(f"{TARGET_COL} and rolling mean")

axes[1].plot(eda_df.index, eda_df[f"{TARGET_COL}_rolling_std"], label="rolling std", lw=1)
axes[1].legend()
axes[1].set_title(f"{TARGET_COL} rolling std ({rolling_window})")

plt.tight_layout()
plt.show()